# Train MatGPT Base Models on a Colab T4

This notebook runs the base-pretraining pipeline for the course validation models:

- `MatGPT-Mini 8M` on TinyStories
- `MatGPT-Tiny 46M` on BabyLM-2026-Strict

Before running: choose **Runtime -> Change runtime type -> T4 GPU**. Colab runtimes are temporary, so this notebook stores runs and checkpoints in Google Drive.

## 1. Choose the run

`smoke` is for checking that the whole pipeline works. `full` is for the serious training run. `resume` continues from the latest checkpoint in Google Drive.

In [ ]:
# @title Run settings
MODEL = "mini_8m"  # @param ["mini_8m", "tiny_46m"]
RUN_MODE = "smoke"  # @param ["smoke", "full", "resume"]

# If you publish this project to GitHub, paste the repo URL here.
# If left blank, the notebook expects PROJECT_DIR to already contain this repo.
REPO_URL = ""  # @param {type:"string"}
PROJECT_DIR = "/content/drive/MyDrive/train-llm-from-scratch"  # @param {type:"string"}

# W&B is optional but recommended for a serious run because it preserves curves
# even if the Colab tab disconnects.
ENABLE_WANDB = True  # @param {type:"boolean"}
WANDB_PROJECT = "matgpt-t4-base"  # @param {type:"string"}
WANDB_ENTITY = ""  # @param {type:"string"}

# A smoke run keeps data and training small. Full runs leave these caps unused.
SMOKE_MAX_TRAIN_DOCS = 5000  # @param {type:"integer"}
SMOKE_MAX_VAL_DOCS = 500  # @param {type:"integer"}
SMOKE_MAX_STEPS = 20  # @param {type:"integer"}

assert MODEL in {"mini_8m", "tiny_46m"}
assert RUN_MODE in {"smoke", "full", "resume"}

## 2. Mount Google Drive

Checkpoints are large and Colab VMs disappear. Drive gives us persistent storage for `runs/`, tokenizer artifacts, and token shards.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Locate or clone the project

This cell makes sure Colab can see the training scripts. If the GitHub repo is private, add a Colab secret named `GITHUB_TOKEN` with repo read access, or make the repository public. If `REPO_URL` is blank, copy this project folder into `PROJECT_DIR` in Drive first.

In [ ]:
import os
import subprocess
from pathlib import Path
from urllib.parse import urlparse, urlunparse

try:
    from google.colab import userdata
except Exception:
    userdata = None

def get_colab_secret(name: str):
    """Read a Colab secret if it exists, otherwise return None."""
    if userdata is None:
        return None
    try:
        return userdata.get(name)
    except Exception:
        return None

def github_clone_url(repo_url: str, token: str | None) -> str:
    """Attach a GitHub token for private HTTPS clones without changing REPO_URL."""
    if not token or not repo_url.startswith("https://github.com/"):
        return repo_url
    parsed = urlparse(repo_url)
    return urlunparse(parsed._replace(netloc=f"x-access-token:{token}@{parsed.netloc}"))

def git_run(args, label: str, token: str | None = None):
    """Run git and show stdout/stderr. This avoids opaque CalledProcessError traces."""
    result = subprocess.run(args, text=True, capture_output=True)
    stdout = result.stdout.replace(token, "<GITHUB_TOKEN>") if token else result.stdout
    stderr = result.stderr.replace(token, "<GITHUB_TOKEN>") if token else result.stderr
    if result.returncode != 0:
        raise RuntimeError(
            f"{label} with exit code {result.returncode}.\n\n"
            f"stdout:\n{stdout}\n\n"
            f"stderr:\n{stderr}\n\n"
            "If this is a private GitHub repo, add a Colab secret named GITHUB_TOKEN "
            "with repo read access, then rerun this cell."
        )
    if stdout:
        print(stdout)
    if stderr:
        print(stderr)
    return result

project_dir = Path(PROJECT_DIR)
github_token = get_colab_secret("GITHUB_TOKEN")

if project_dir.exists() and (project_dir / ".git").exists():
    print("Existing git checkout found. Pulling latest changes.")
    git_pull = git_run(["git", "-C", str(project_dir), "pull", "--ff-only"], "git pull", github_token)
elif REPO_URL:
    if project_dir.exists() and any(project_dir.iterdir()):
        raise RuntimeError(
            f"PROJECT_DIR exists but is not a git checkout: {project_dir}\n"
            "Choose a different PROJECT_DIR, delete the partial folder, or copy the repo there manually."
        )
    project_dir.parent.mkdir(parents=True, exist_ok=True)
    clone_url = github_clone_url(REPO_URL, github_token)
    git_run(["git", "clone", clone_url, str(project_dir)], "Git clone failed", github_token)
elif not project_dir.exists():
    raise RuntimeError(
        "REPO_URL is blank and PROJECT_DIR does not exist. Set REPO_URL to the GitHub repo, "
        "or copy the project folder into Drive first."
    )

os.chdir(project_dir)
print("Current directory:", Path.cwd())
assert Path("scripts/pretrain.py").exists(), "PROJECT_DIR must point at the MatGPT repository root."

## 4. Install dependencies

This installs the local package, dataset tools, W&B, and optional Colab helpers such as bitsandbytes.

In [ ]:
%pip install -q -e ".[test,colab]" huggingface_hub wandb

## 5. Authenticate Hugging Face and W&B

Recommended Colab secrets:

- `HF_TOKEN`: Hugging Face token for dataset access
- `WANDB_API_KEY`: Weights & Biases API key

If a secret is missing, the notebook falls back to the normal interactive login prompt.

In [ ]:
from huggingface_hub import login as hf_login
import wandb

try:
    from google.colab import userdata
except Exception:
    userdata = None

def get_colab_secret(name: str):
    """Read a Colab secret if it exists, otherwise return None."""
    if userdata is None:
        return None
    try:
        return userdata.get(name)
    except Exception:
        return None

hf_token = get_colab_secret("HF_TOKEN")
if hf_token:
    hf_login(token=hf_token, add_to_git_credential=False)
else:
    hf_login(add_to_git_credential=False)

if ENABLE_WANDB:
    wandb_key = get_colab_secret("WANDB_API_KEY")
    if wandb_key:
        wandb.login(key=wandb_key)
    else:
        wandb.login()
else:
    print("W&B disabled for this run.")

## 6. Check the GPU

A free Colab runtime is not guaranteed to be a T4 every time. Confirm the GPU before starting a long run.

In [ ]:
!nvidia-smi

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 7. Build a Colab-specific config

We start from the repo config, then change only runtime details: output paths, W&B settings, and smoke-run caps. Full runs do not cap the dataset.

In [ ]:
import json
import yaml
from pathlib import Path

base_config_path = Path("configs/matgpt_mini_8m.yaml") if MODEL == "mini_8m" else Path("configs/matgpt_tiny_46m.yaml")
cfg = yaml.safe_load(base_config_path.read_text())

drive_root = Path("/content/drive/MyDrive/matgpt_runs") / cfg["run"]["name"]
cfg["run"]["output_dir"] = str(drive_root / "run")
cfg["dataset"]["normalized_dir"] = str(drive_root / "normalized")
cfg["tokenizer"]["output_dir"] = str(drive_root / "tokenizer")
cfg["sharding"]["output_dir"] = str(drive_root / "shards")

cfg["tracking"]["wandb"]["enabled"] = bool(ENABLE_WANDB)
cfg["tracking"]["wandb"]["project"] = WANDB_PROJECT
cfg["tracking"]["wandb"]["entity"] = WANDB_ENTITY or None

if RUN_MODE == "smoke":
    cfg["dataset"]["max_train_documents"] = SMOKE_MAX_TRAIN_DOCS
    cfg["dataset"]["max_validation_documents"] = SMOKE_MAX_VAL_DOCS
    cfg["training"]["max_tokens"] = 200_000
    cfg["training"]["eval_batches"] = 8
    cfg["training"]["eval_interval_tokens"] = 50_000
    cfg["training"]["sample_interval_tokens"] = 50_000
    cfg["training"]["checkpoint_interval_tokens"] = 50_000

colab_config_dir = Path("configs/colab")
colab_config_dir.mkdir(parents=True, exist_ok=True)
colab_config_path = colab_config_dir / f"{MODEL}_{RUN_MODE}.yaml"
colab_config_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding="utf-8")

print("Wrote", colab_config_path)
print(json.dumps({
    "model": MODEL,
    "run_mode": RUN_MODE,
    "output_dir": cfg["run"]["output_dir"],
    "wandb_enabled": cfg["tracking"]["wandb"]["enabled"],
}, indent=2))

## 8. Prepare data, tokenizer, and shards

These three steps are separate on purpose:

1. Normalize and manifest the text corpus.
2. Train the tokenizer only on training text.
3. Tokenize once into packed binary shards so training does not waste time re-tokenizing.

In [ ]:
import subprocess
import shlex

def run_command(command):
    """Run a command and show stdout/stderr when it fails.

    Colab otherwise reports only `CalledProcessError`, which hides the real
    Python traceback from scripts such as `pretrain.py`.
    """
    if isinstance(command, list):
        printable = " ".join(shlex.quote(str(part)) for part in command)
    else:
        printable = command
    print("\n$", printable)
    result = subprocess.run(command, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code {result.returncode}: {printable}\n\n"
            f"stdout:\n{result.stdout}\n\n"
            f"stderr:\n{result.stderr}"
        )

run_command(["python", "scripts/prepare_dataset.py", "--config", str(colab_config_path)])
run_command(["python", "scripts/train_tokenizer.py", "--config", str(colab_config_path)])
run_command(["python", "scripts/tokenize_and_shard.py", "--config", str(colab_config_path)])

## 9. Benchmark the T4

Use this before full training. If a larger micro-batch fails, keep the config conservative or lower `micro_batch_size` and increase `gradient_accumulation_steps` to keep the effective token batch similar.

In [ ]:
batch_sizes = "8,16,24,32" if MODEL == "mini_8m" else "2,4,6,8"
run_command(["python", "scripts/benchmark_t4.py", "--config", str(colab_config_path), "--batch-sizes", batch_sizes, "--steps", "5"])

## 10. Train or resume

For `resume`, the notebook loads `latest.pt` from the configured Drive checkpoint directory. For `smoke`, it runs only `SMOKE_MAX_STEPS` even if the config has a larger token target.

In [ ]:
train_cmd = ["python", "scripts/pretrain.py", "--config", str(colab_config_path)]

if RUN_MODE == "smoke":
    train_cmd += ["--max-steps", str(SMOKE_MAX_STEPS)]

if RUN_MODE == "resume":
    latest = Path(cfg["run"]["output_dir"]) / "checkpoints" / "latest.pt"
    assert latest.exists(), f"No checkpoint found at {latest}"
    train_cmd += ["--resume-from", str(latest)]

run_command(train_cmd)

## 11. Evaluate the latest or best checkpoint

Validation loss and perplexity tell you whether the model is improving. Fixed prompt samples show whether generations are becoming more coherent.

In [ ]:
checkpoint_dir = Path(cfg["run"]["output_dir"]) / "checkpoints"
best = checkpoint_dir / "best.pt"
latest = checkpoint_dir / "latest.pt"
checkpoint = best if best.exists() else latest
assert checkpoint.exists(), "No checkpoint found. Run training first."

run_command(["python", "scripts/evaluate.py", "--config", str(colab_config_path), "--checkpoint", str(checkpoint)])

## 12. Inspect metrics and samples

The CSV is the local record. W&B is the remote dashboard. Use both: CSV is simple and reproducible, W&B is better for live monitoring.

In [ ]:
import pandas as pd
from IPython.display import display

metrics_path = Path(cfg["run"]["output_dir"]) / "metrics.csv"
if metrics_path.exists():
    metrics = pd.read_csv(metrics_path)
    display(metrics.tail(20))
    for column in ["train_loss", "val_loss"]:
        if column in metrics.columns:
            metrics.dropna(subset=[column]).plot(x="tokens_processed", y=column, title=column)
else:
    print("No metrics.csv found yet.")

sample_dir = Path(cfg["run"]["output_dir"]) / "samples"
if sample_dir.exists():
    sample_files = sorted(sample_dir.glob("samples_*.json"))
    if sample_files:
        print("Latest sample file:", sample_files[-1])
        print(sample_files[-1].read_text()[:4000])

## 13. Generate from the checkpoint

This uses the base model as a text completer. It is not a chatbot yet; SFT and DPO come later.

In [ ]:
prompt = "Once upon a time" if MODEL == "mini_8m" else "A token is"
run_command([
    "python", "scripts/chat.py",
    "--config", str(colab_config_path),
    "--checkpoint", str(checkpoint),
    "--prompt", prompt,
    "--max-new-tokens", "120",
])